## This will fill up the resident_email column as they were missing using the resident_name we get to know their email 

In [8]:
import pandas as pd
import numpy as np
import random

# Load the files

new_data = pd.read_csv('civic_complaints_100_v4.csv')

#  Apply your email logic
def create_email(name, is_anonymous):
    if is_anonymous or pd.isna(name):
        return np.nan
    return f"{name.lower().replace(' ', '.')}@{random.choice(['gmail.com', 'outlook.com', 'yahoo.com'])}"

new_data['resident_email'] = new_data.apply(lambda row: create_email(row['resident_name'], row['is_anonymous']), axis=1)

# #  Merge into the original structure
# # Ensure your columns match exactly as defined in your project report 
# updated_df = pd.concat([original_df, new_data]).drop_duplicates(subset=['report_id'])

# # Save the finalized, filled dataset
new_data.to_csv('civic_complaints_100_v4.csv', index=False)

## This will append 4 columns because our initial dataset has missing columns so we need them for making priority : 
### 'priority_level','priority_score' , 'severity_score', 'area_importance', 'citizen_reports_count'
### and i used formulas to fill them up and different technique you can check the code 

In [7]:
import pandas as pd
import random

original_df = pd.read_csv('civic-issue-reporting-dataset.csv')

def infer_and_calculate_priority(row):
    desc = str(row['issue_description']).lower()
    loc = str(row['location_street_address']).lower()
    
    # 1. Infer Severity Score (1-10)
    if any(word in desc for word in ['accident', 'dangerous', 'injured', 'flooding', 'fire', 'collapse']):
        severity = random.randint(8, 10)
    elif any(word in desc for word in ['overflowing', 'broken', 'large', 'not working']):
        severity = random.randint(5, 7)
    else:
        severity = random.randint(1, 4)
        
    # 2. Infer Area Importance (1-10)
    if 'hospital' in loc: area_imp = 10
    elif 'school' in loc: area_imp = 9
    elif 'highway' in loc or 'main' in loc: area_imp = 8
    elif 'commercial' in loc or 'st' in loc: area_imp = 7
    elif 'park' in loc: area_imp = 5
    else: area_imp = random.randint(2, 3)
    
    # 3. Citizen Reports Count (Random simulation)
    count = random.randint(1, 50)
    
    # 4. Apply your exact Formula
    norm_count = (count / 50) * 10
    priority_score = (0.4 * severity) + (0.3 * norm_count) + (0.3 * area_imp)
    
    # 5. Map Priority Level
    if priority_score < 4.0: level = 'low'
    elif priority_score < 7.0: level = 'medium'
    elif priority_score < 8.5: level = 'high'
    else: level = 'critical'
    
    return pd.Series([level, round(priority_score, 2), severity, area_imp, count])

# Apply to your original dataframe
cols = ['priority_level','priority_score' , 'severity_score', 'area_importance', 'citizen_reports_count' ]
new_data = original_df.apply(infer_and_calculate_priority, axis=1)
new_data.columns = cols

# 3. Concatenate horizontally
final_df = pd.concat([original_df, new_data], axis=1)

# 4. Save
final_df.to_csv('civic-issue-reporting-dataset.csv', index=False)

### droping the report_id and resident_id becasue of improper format and again fills 

In [2]:
import pandas as pd
import random

df = pd.read_csv("civic-issue-reporting-dataset.csv")

# Drop old IDs
df = df.drop(columns=['report_id', 'resident_id'], errors='ignore')

#   Create report_id (always unique per row)


# Generate unique random report IDs
report_ids = random.sample(range(100000, 999999), len(df))
df['report_id'] = [f"RPT-{i}" for i in report_ids]

#  Create resident_id using mapping (IMPORTANT)
unique_residents = df['resident_name'].unique()

random_ids = random.sample(range(1000, 9999), len(unique_residents))

mapping = {name: f"RES-{rid}" for name, rid in zip(unique_residents, random_ids)}

df['resident_id'] = df['resident_name'].map(mapping)

# Put these columns first
cols = ['report_id', 'resident_id'] + [col for col in df.columns if col not in ['report_id', 'resident_id']]

df = df[cols]


df.to_csv("civic-issue-reporting-dataset.csv", index=False)

print(df.head())

    report_id resident_id     resident_name                resident_email  \
0  RPT-814592    RES-1628  Charlotte Brooks  charlotte.brooks94@gmail.com   
1  RPT-115040    RES-6964               NaN                           NaN   
2  RPT-439214    RES-2576    Miguel Sanchez        msanchez77@outlook.com   
3  RPT-907060    RES-8893       Jasmine Lee              jlee98@gmail.com   
4  RPT-440214    RES-6964               NaN                           NaN   

    resident_phone   issue_type  \
0  +1-212-555-0896      pothole   
1              NaN        noise   
2  +1 718 555 2390      garbage   
3  +1-917-555-0147  streetlight   
4              NaN     graffiti   

                                   issue_description issue_status  \
0  Large pothole causing traffic disruption near ...  in-progress   
1  Construction noise persists late into the nigh...    submitted   
2  Overflowing garbage bins on the corner are att...     resolved   
3  Streetlight not functioning, resulting in a da.

In [4]:
import pandas as pd
import random

df = pd.read_csv("civic_complaints_100_v4.csv")

# clean names
df['resident_name'] = df['resident_name'].str.strip().str.lower()

# Step 1: build mapping ONLY from existing IDs (without dropping anything)
existing_mapping = df.dropna(subset=['resident_id'])
name_to_id = dict(zip(existing_mapping['resident_name'], existing_mapping['resident_id']))

# Step 2: find missing resident IDs
missing_mask = df['resident_id'].isna()
missing_names = df.loc[missing_mask, 'resident_name'].unique()

# Step 3: generate new IDs for missing people
random_ids = random.sample(range(1000, 9999), len(missing_names))
new_mapping = {name: f"RES-{rid}" for name, rid in zip(missing_names, random_ids)}

# Step 4: combine mappings
name_to_id.update(new_mapping)

# Step 5: fill ALL rows (no dropping, no loss)
df['resident_id'] = df['resident_name'].map(name_to_id)

df.to_csv("civic_complaints_100_v4.csv", index=False)

df.head()

,report_id,resident_id,resident_name,resident_email,resident_phone,issue_type,issue_description,issue_status,report_datetime,location_street_address,...,photo_url,municipal_department,resolution_datetime,resolution_notes,is_anonymous,priority_level,priority_score,severity_score,area_importance,citizen_reports_count
0,RPT-9651E,RES-1905,NaN,NaN,NaN,pothole,The road is falling apart here.,resolved,2025-05-27T14:27:23.739095,589 Main St,...,NaN,Roads & Highways,2025-05-30T14:27:23.739095,Issue addressed and closed.,True,low,3.84,3,2,34
1,RPT-4621F,RES-1495,mary taylor,mary.taylor@outlook.com,+1-237-555-2461,noise,Barking dogs kept for hours.,submitted,2026-01-21T14:27:23.739326,428 Main St,...,NaN,Public Safety,NaN,NaN,False,medium,5.88,3,6,48
2,RPT-2350A,RES-4613,mary hernandez,mary.hernandez@gmail.com,+1-357-555-5749,drainage,Drain cover is broken.,in-progress,2025-05-06T14:27:23.739378,709 Main St,...,https://civicphotos.gov.us/drainage-RPT-2350A.jpg,Drainage Authority,NaN,NaN,False,medium,5.64,6,9,9
3,RPT-5462F,RES-1905,NaN,NaN,NaN,pothole,Pothole is expanding every day.,in-progress,2025-12-15T14:27:23.739411,388 Main St,...,NaN,Roads & Highways,NaN,NaN,True,high,7.18,10,8,13
4,RPT-1468F,RES-1905,NaN,NaN,NaN,drainage,Smell coming from the drain.,submitted,2025-12-18T14:27:23.739441,537 Main St,...,NaN,Drainage Authority,NaN,NaN,True,high,8.44,10,9,29


In [5]:
df = pd.read_csv('civic_complaints_100_v4.csv')

count = df.isnull().sum()

print(count)

report_id                   0
resident_id                 0
resident_name              48
resident_email             48
resident_phone             48
issue_type                  0
issue_description           0
issue_status                0
report_datetime             0
location_street_address     0
location_city               0
location_state              0
location_postal_code        0
location_country            0
latitude                    0
longitude                   0
photo_url                  60
municipal_department        0
resolution_datetime        68
resolution_notes           68
is_anonymous                0
priority_level              0
priority_score              0
severity_score              0
area_importance             0
citizen_reports_count       0
dtype: int64


In [14]:
df = pd.read_csv('municipal_complaints_v2.csv')


missing = df.isnull().sum()

print(missing)

print(df.shape)

report_id                    0
resident_id                  0
resident_name               63
resident_email              63
resident_phone              78
issue_type                   0
issue_description            0
issue_status                 0
report_datetime              0
location_street_address      0
location_city                0
location_state               0
location_postal_code         0
location_country             0
latitude                     0
longitude                    0
photo_url                  419
municipal_department         7
resolution_datetime        430
resolution_notes           430
is_anonymous                 0
priority_level               0
priority_score               0
severity_score               0
area_importance              0
citizen_reports_count        0
dtype: int64
(700, 26)


In [ ]:
# high_priority_descriptions = [
#     # existing ones
#     "dangerous flooding blocking road completely",
#     "fire accident near building collapse reported",
#     "injured pedestrian dangerous pothole on highway",
#     "overflowing sewage flooding entire street dangerous",
#     # ADD THESE for critical
#     "building collapse dangerous emergency evacuation needed",
#     "fire spreading rapidly accident injured people reported",
#     "dangerous flooding entire area collapsed infrastructure",
#     "multiple injured accident dangerous road collapse reported",
#     "emergency fire accident hospital road completely blocked",
#     "sewage flooding dangerous collapse near school highway",
# ]

# high_priority_locations = [
#     # existing ones
#     "hospital road main street",
#     "school highway commercial district",
#     "main hospital avenue",
#     # ADD THESE
#     "main hospital highway",
#     "school main road",
#     "highway hospital commercial street",
# ]

In [ ]:
# import pandas as pd
# import random

# original_df = pd.read_csv('municipal_complaints_v2.csv')

# # Drop the old calculated columns from original_df first
# cols_to_recalculate = ['priority_level', 'priority_score', 'severity_score', 
#                         'area_importance', 'citizen_reports_count']

# original_df = original_df.drop(columns=cols_to_recalculate, errors='ignore')  # ✅

# # Now rest of your script runs without duplicates

# high_priority_descriptions = [
#     # existing ones
#     "dangerous flooding blocking road completely",
#     "fire accident near building collapse reported",
#     "injured pedestrian dangerous pothole on highway",
#     "overflowing sewage flooding entire street dangerous",
#     # ADD THESE for critical
#     "building collapse dangerous emergency evacuation needed",
#     "fire spreading rapidly accident injured people reported",
#     "dangerous flooding entire area collapsed infrastructure",
#     "multiple injured accident dangerous road collapse reported",
#     "emergency fire accident hospital road completely blocked",
#     "sewage flooding dangerous collapse near school highway",
# ]

# high_priority_locations = [
#     # existing ones
#     "hospital road main street",
#     "school highway commercial district",
#     "main hospital avenue",
#     # ADD THESE
#     "main hospital highway",
#     "school main road",
#     "highway hospital commercial street",
# ]



# def infer_and_calculate_priority(row):
#     desc = str(row['issue_description']).lower()
#     loc = str(row['location_street_address']).lower()
    
#     # 1. Infer Severity Score (1-10)
#     if any(word in desc for word in ['accident', 'dangerous', 'injured', 'flooding', 'fire', 'collapse']):
#         severity = random.randint(8, 10)
#     elif any(word in desc for word in ['overflowing', 'broken', 'large', 'not working']):
#         severity = random.randint(5, 7)
#     else:
#         severity = random.randint(1, 4)
        
#     # 2. Infer Area Importance (1-10)
#     if 'hospital' in loc: area_imp = 10
#     elif 'school' in loc: area_imp = 9
#     elif 'highway' in loc or 'main' in loc: area_imp = 8
#     elif 'commercial' in loc or 'st' in loc: area_imp = 7
#     elif 'park' in loc: area_imp = 5
#     else: area_imp = random.randint(2, 3)
    
#     # 3. Citizen Reports Count (Random simulation)
#     count = random.randint(1, 50)
    
#     # 4. Apply Formula
#     norm_count = (count / 50) * 10
#     priority_score = (0.4 * severity) + (0.3 * norm_count) + (0.3 * area_imp)
    
#     # 5. Map Priority Level
#     if priority_score < 4.0: level = 'low'
#     elif priority_score < 6.0: level = 'medium'
#     elif priority_score < 7.5: level = 'high'
#     else: level = 'critical'
    
#     return pd.Series([level, round(priority_score, 2), severity, area_imp, count])

# # Apply to original dataframe
# cols = ['priority_level', 'priority_score', 'severity_score', 'area_importance', 'citizen_reports_count']
# new_data = original_df.apply(infer_and_calculate_priority, axis=1)
# new_data.columns = cols
# final_df = pd.concat([original_df, new_data], axis=1)

# # ADD THIS BLOCK HERE (generate 300 extra high-priority rows)
# extra_rows = []
# for _ in range(400):
#     row = original_df.sample(1).iloc[0].copy()
#     row['issue_description'] = random.choice(high_priority_descriptions)
#     row['location_street_address'] = random.choice(high_priority_locations)
#     extra_rows.append(row)

# extra_df = pd.DataFrame(extra_rows).reset_index(drop=True) 
# extra_scores = extra_df.apply(infer_and_calculate_priority, axis=1)
# extra_scores.columns = cols
# extra_scores = extra_scores.reset_index(drop=True)            

# extra_df = pd.concat([extra_df, extra_scores], axis=1)

# # COMBINE EVERYTHING
# final_df = pd.concat([final_df, extra_df], ignore_index=True)

# # Save
# final_df.to_csv('second.csv', index=False)

# # CHECK YOUR CLASS DISTRIBUTION
# print(final_df['priority_level'].value_counts())

priority_level
low         292
medium      289
critical    289
high        230
Name: count, dtype: int64


In [ ]:
# import pandas as pd
# import random

# # 1. Load original
# original_df = pd.read_csv('municipal_complaints_v2.csv')

# # 2. Load your 4 Gemini batches
# batch1 = pd.read_csv('batch1_high.csv')
# batch2 = pd.read_csv('batch2_high.csv')
# batch3 = pd.read_csv('batch3_critical.csv')
# batch4 = pd.read_csv('batch4_critical.csv')

# # 3. Stack all batches together
# gemini_df = pd.concat([batch1, batch2, batch3, batch4], ignore_index=True)

# # 4. Run your priority formula on gemini rows
# def infer_and_calculate_priority(row):
#     desc = str(row['issue_description']).lower()
#     loc = str(row['location_street_address']).lower()
    
#     if any(word in desc for word in ['accident', 'dangerous', 'injured', 'flooding', 'fire', 'collapse']):
#         severity = random.randint(8, 10)
#     elif any(word in desc for word in ['overflowing', 'broken', 'large', 'not working']):
#         severity = random.randint(5, 7)
#     else:
#         severity = random.randint(1, 4)
        
#     if 'hospital' in loc: area_imp = 10
#     elif 'school' in loc: area_imp = 9
#     elif 'highway' in loc or 'main' in loc: area_imp = 8
#     elif 'commercial' in loc or 'st' in loc: area_imp = 7
#     elif 'park' in loc: area_imp = 5
#     else: area_imp = random.randint(2, 3)
    
#     count = random.randint(1, 50)
#     norm_count = (count / 50) * 10
#     priority_score = (0.4 * severity) + (0.3 * norm_count) + (0.3 * area_imp)
    
#     if priority_score < 4.0: level = 'low'
#     elif priority_score < 6.0: level = 'medium'
#     elif priority_score < 7.5: level = 'high'
#     else: level = 'critical'
    
#     return pd.Series([level, round(priority_score, 2), severity, area_imp, count])

# cols = ['priority_level', 'priority_score', 'severity_score', 
#         'area_importance', 'citizen_reports_count']

# # 5. Apply formula to gemini rows only
# gemini_scores = gemini_df.apply(infer_and_calculate_priority, axis=1)
# gemini_scores.columns = cols
# gemini_df = pd.concat([gemini_df.reset_index(drop=True), 
#                        gemini_scores.reset_index(drop=True)], axis=1)

# # 6. Combine with original
# final_df = pd.concat([original_df, gemini_df], ignore_index=True)

# # 7. SHUFFLE 
# final_df = final_df.sample(frac=1, random_state=42).reset_index(drop=True)

# # 8. Save
# final_df.to_csv('final_dataset.csv', index=False)

# # 9. Check distribution
# print(final_df['priority_level'].value_counts())
# print(f"Total rows: {len(final_df)}")

In [1]:
import pandas as pd

# 1. Load your original 700-row file
# Replace 'my_original_700.csv' with your actual filename
df_original = pd.read_csv('municipal_complaints_v2.csv')

# 2. Load the 4 new batches
b1 = pd.read_csv('Final_Batch_1_High.csv')
b2 = pd.read_csv('Final_Batch_2_High.csv')
b3 = pd.read_csv('Final_Batch_3_Critical.csv')
b4 = pd.read_csv('Final_Batch_4_Critical.csv')

# 3. Combine all dataframes into one (Total: 1100 rows)
df_combined = pd.concat([df_original, b1, b2, b3, b4], ignore_index=True)

# 4. SHUFFLE the rows
# frac=1 tells pandas to return 100% of the rows in a random order
# random_state=42 (optional) makes the shuffle repeatable for debugging
df_shuffled = df_combined.sample(frac=1, random_state=42).reset_index(drop=True)

# 5. Save your final training dataset
df_shuffled.to_csv('municipal_training_set_1100.csv', index=False)

print(f"Successfully shuffled {len(df_shuffled)} rows into a new file!")

Successfully shuffled 1100 rows into a new file!


In [2]:
df = pd.read_csv('municipal_training_set_1100.csv')


missing = df.isnull().sum()

print(missing)

print(df.shape)

report_id                    0
resident_id                  0
resident_name               63
resident_email              63
resident_phone              78
issue_type                   0
issue_description            0
issue_status                 0
report_datetime              0
location_street_address      0
location_city                0
location_state               0
location_postal_code         0
location_country             0
latitude                     0
longitude                    0
photo_url                  454
municipal_department         7
resolution_datetime        830
resolution_notes           830
is_anonymous                 0
priority_level               0
priority_score               0
severity_score               0
area_importance              0
citizen_reports_count        0
dtype: int64
(1100, 26)
